In [1]:
!pip install -q "vllm==0.6.3.post1" "datasets" "sacrebleu" "unbabel-comet" pandas tqdm

In [2]:
import os
import gc
import time
import pandas as pd
import torch
from tqdm.auto import tqdm

MODELS = {
    "glm-4": "ModelCloud/glm-4-9b-gptq-4bit",
    "qwen2.5": "Qwen/Qwen2.5-7B-Instruct-AWQ",
}

DIRECTIONS = [
    ("eng_Latn", "cmn_Hans", "English -> Mandarin"),
    ("cmn_Hans", "eng_Latn", "Mandarin -> English"),
    ("eng_Latn", "yue_Hant", "English -> Cantonese"),
    ("yue_Hant", "eng_Latn", "Cantonese -> English"),
]

N_SAMPLES = 100
MAX_NEW_TOKENS = 256

In [3]:
from vllm import LLM, SamplingParams

def unload_model(llm):
    del llm
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
from datasets import load_dataset

langs = ["eng_Latn", "cmn_Hans", "yue_Hant"]
flores = {}
for lang in langs:
    flores[lang] = load_dataset("openlanguagedata/flores_plus", lang, split="devtest")

df = pd.DataFrame({
    "id": range(len(flores["eng_Latn"])),
    "eng_Latn": flores["eng_Latn"]["text"],
    "cmn_Hans": flores["cmn_Hans"]["text"],
    "yue_Hant": flores["yue_Hant"]["text"],
}).head(N_SAMPLES)

df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/227 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/221 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/227 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/221 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/227 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/221 [00:00<?, ?it/s]

,id,eng_Latn,cmn_Hans,yue_Hant
0,0,"""We now have 4-month-old mice that are non-dia...",他补充道：“我们现在有 4 个月大没有糖尿病的老鼠，但它们曾经得过该病。”,佢補充話：「我哋而家有 4 個月大嘅小鼠，佢哋而家冇糖尿病，但曾經有過糖尿病。」
1,1,"Dr. Ehud Ur, professor of medicine at Dalhousi...",埃胡德·乌尔博士（新斯科舍省哈利法克斯市达尔豪西大学医学教授，加拿大糖尿病协会临床与科学部门...,新斯科舍省哈利法克斯嘅達爾豪西大學醫學教授兼加拿大糖尿病協會臨床與科學部門主席艾胡德·烏爾醫...
2,2,"Like some other experts, he is skeptical about...",和其他一些专家一样，他对糖尿病能否治愈持怀疑态度。他指出，这些发现与已患有 1 型糖尿病的人无关。,同其他專家一樣，佢對於係咪可以醫好糖尿病抱有懷疑態度，而且指出呢項發現同1型糖尿病病人冇關。
3,3,"On Monday, Sara Danius, permanent secretary of...",周一，瑞典学院诺贝尔文学委员会常务秘书萨拉·丹尼尔斯在瑞典广播电台的一档节目中向公众宣布，委...,喺星期一，瑞典學院諾貝爾文學獎委員會常任秘書薩拉·丹尼烏斯喺瑞典廣播電台嘅一個廣播節目入面公...
4,4,"Danius said, ""Right now we are doing nothing. ...",达尼厄斯说道：“目前我们保持按兵不动。我给他关系最好的合作者打过电话并发送了电子邮件，而且收...,丹尼烏斯話：「而家我哋唔採取任何行動。我已經打咗電話同發電郵畀佢最緊密嘅拍檔，而且得到非常友...


In [5]:
CANTONESE_MARKERS = ["嘅", "咗", "喺", "唔", "佢", "啲", "嘢", "乜"]

def marker_count(text):
    return sum(text.count(m) for m in CANTONESE_MARKERS)

df["yue_hant_cantonese_markers"] = df["yue_Hant"].apply(marker_count)

pct_with_markers = (df["yue_hant_cantonese_markers"] > 0).mean() * 100
print(f"% of yue_Hant reference sentences containing at least one Cantonese-specific marker: {pct_with_markers:.1f}%")
print()
print("If this is low (well under ~50%), the reference set is likely still the known")
print("Mandarin-in-Traditional-script bug -- treat Cantonese-direction scores accordingly.")
df.sort_values("yue_hant_cantonese_markers", ascending=False).head(5)[["yue_Hant", "yue_hant_cantonese_markers"]]

% of yue_Hant reference sentences containing at least one Cantonese-specific marker: 85.0%

If this is low (well under ~50%), the reference set is likely still the known
Mandarin-in-Traditional-script bug -- treat Cantonese-direction scores accordingly.


,yue_Hant,yue_hant_cantonese_markers
31,羽毛嘅結構表明佢哋並唔係用於飛行，而係調節溫度或者展示。 研究人員認為，就算係一隻年幼恐龍嘅...,8
36,佢哋發現太陽嘅運行原理同其他恆星相同：系統中嘅所有恆星嘅活動都係由佢哋嘅光度、自轉驅動嘅，而...,8
86,「佢[威爾士]基本上從一開始就對我哋講大話。首先，佢表現呢個動作係出於法律原因。第二，佢扮嗮...,7
11,一種名為 ZMapp 嘅「抗體雞尾酒」最初有望喺呢個領域發揮作用，但正式研究顯示，佢喺預防死...,7
27,佢唔單止證實咗至少有一啲恐龍係有羽毛（呢種講法已經廣為流傳），而且提供咗化石通常無法提供嘅細...,6


In [6]:
LANG_NAME = {
    "eng_Latn": "English",
    "cmn_Hans": "Mandarin Chinese (Simplified script)",
    "yue_Hant": "Cantonese (Yue Chinese, Traditional script)",
}

def build_prompt(text, src, tgt):
    return (
        f"Translate the following {LANG_NAME[src]} text into {LANG_NAME[tgt]}. "
        f"Output only the translation, nothing else.\n\n{text}"
    )

In [ ]:
### GLM-4 generation — notes on model choice and settings

Two decisions here are non-obvious and worth documenting for anyone re-running this:

**Why `ModelCloud/glm-4-9b-gptq-4bit` instead of `THUDM/glm-4-9b-chat`:** vLLM's on-the-fly
bitsandbytes quantization doesn't support ChatGLM's architecture in this vLLM version
(`AttributeError: Model ChatGLMForCausalLM does not support BitsAndBytes quantization yet.`).
This community GPTQ checkpoint is quantized on disk already, which sidesteps that gap and
also uses much less peak memory during load.

**Why there's no `stop_token_ids` override:** an earlier attempt using GLM-4-9B-chat's official
stop tokens (`151329, 151336, 151338`) caused ~38% of generations to come back empty --
those IDs don't line up correctly with this community checkpoint's tokenizer. Letting the
tokenizer's own EOS handling take over fixed it completely (0 empty outputs).

Caveat for the recommendation doc: these are third-party quantized weights, not vendor-released
ones -- worth flagging if GLM-4 is ever reconsidered against its official checkpoint.

In [8]:
llm = LLM(
    model="ModelCloud/glm-4-9b-gptq-4bit",
    trust_remote_code=True,
    quantization="gptq",
    dtype="float16",
    max_model_len=1024,
    gpu_memory_utilization=0.85,
    enforce_eager=True,
)
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(temperature=0, max_tokens=MAX_NEW_TOKENS)  # no stop_token_ids override -- see markdown note above

glm_results = []
for src, tgt, label in DIRECTIONS:
    print(f"  glm-4 — {label}")
    chat_prompts = []
    for _, row in df.iterrows():
        prompt = build_prompt(row[src], src, tgt)
        messages = [{"role": "user", "content": prompt}]
        chat_prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    outputs = llm.generate(chat_prompts, sampling_params)
    for (_, row), output in zip(df.iterrows(), outputs):
        hyp = output.outputs[0].text.strip()
        glm_results.append({
            "id": row["id"], "direction": label, "src_lang": src, "tgt_lang": tgt,
            "model": "glm-4", "source": row[src], "reference": row[tgt], "hypothesis": hyp,
        })

unload_model(llm)
glm_df_raw = pd.DataFrame(glm_results)
print("Empty/null hypotheses:", (glm_df_raw["hypothesis"].str.strip() == "").sum(), "/", len(glm_df_raw))
glm_df_raw.to_csv("glm_only_retry_vllm.csv", index=False)

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 07-16 01:17:21 config.py:321] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 07-16 01:17:21 config.py:395] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 07-16 01:17:21 llm_engine.py:237] Initializing an LLM engine (v0.6.3.post1) with config: model='ModelCloud/glm-4-9b-gptq-4bit', speculative_config=None, tokenizer='ModelCloud/glm-4-9b-gptq-4bit', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=gptq, enforce_eager=True, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guid

/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 07-16 01:17:23 model_runner.py:1056] Starting to load model ModelCloud/glm-4-9b-gptq-4bit...
INFO 07-16 01:17:24 selector.py:224] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 07-16 01:17:24 selector.py:115] Using XFormers backend.
INFO 07-16 01:17:24 weight_utils.py:243] Using model weights format ['*.safetensors']
INFO 07-16 01:17:24 weight_utils.py:288] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


...

  glm-4 (retry) — English -> Cantonese


Processed prompts: 100%|██████████| 100/100 [00:25<00:00,  3.97it/s, est. speed input: 237.04 toks/s, output: 130.36 toks/s]


  glm-4 (retry) — Cantonese -> English


Processed prompts: 100%|██████████| 100/100 [00:15<00:00,  6.46it/s, est. speed input: 491.47 toks/s, output: 137.42 toks/s]


Nulls after retry: 0 / 400


In [8]:
qwen_results = []
model_id = "Qwen/Qwen2.5-7B-Instruct-AWQ"
print(f"Loading qwen2.5 ({model_id}) via vLLM...")

llm = LLM(
    model=model_id,
    trust_remote_code=True,
    quantization="awq",
    dtype="float16",
    max_model_len=512,
    gpu_memory_utilization=0.55,
    enforce_eager=True,
)
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(temperature=0, max_tokens=MAX_NEW_TOKENS)

for src, tgt, label in DIRECTIONS:
    print(f"  qwen2.5 — {label}")
    chat_prompts = []
    for _, row in df.iterrows():
        prompt = build_prompt(row[src], src, tgt)
        messages = [{"role": "user", "content": prompt}]
        chat_prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    outputs = llm.generate(chat_prompts, sampling_params)
    for (_, row), output in zip(df.iterrows(), outputs):
        hyp = output.outputs[0].text.strip()
        qwen_results.append({
            "id": row["id"], "direction": label, "src_lang": src, "tgt_lang": tgt,
            "model": "qwen2.5", "source": row[src], "reference": row[tgt], "hypothesis": hyp,
        })

unload_model(llm)
qwen_only_df = pd.DataFrame(qwen_results)
qwen_only_df.to_csv("qwen_only_vllm.csv", index=False)  # save immediately, before doing anything else
print("Qwen2.5 nulls:", qwen_only_df["hypothesis"].isna().sum(), "/", len(qwen_only_df))
qwen_only_df.head()

Loading qwen2.5 (Qwen/Qwen2.5-7B-Instruct-AWQ) via vLLM...


`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 07-16 01:24:47 config.py:321] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 07-16 01:24:47 config.py:395] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 07-16 01:24:47 llm_engine.py:237] Initializing an LLM engine (v0.6.3.post1) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=True, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_de

/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 07-16 01:24:49 model_runner.py:1056] Starting to load model Qwen/Qwen2.5-7B-Instruct-AWQ...
INFO 07-16 01:24:49 selector.py:224] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 07-16 01:24:49 selector.py:115] Using XFormers backend.
INFO 07-16 01:24:49 weight_utils.py:243] Using model weights format ['*.safetensors']


...

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 07-16 01:27:10 scheduler.py:1483] Sequence group 326 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=101


Processed prompts: 100%|██████████| 100/100 [00:41<00:00,  2.42it/s, est. speed input: 224.09 toks/s, output: 81.19 toks/s]


Qwen2.5 nulls: 0 / 400


,id,direction,src_lang,tgt_lang,model,source,reference,hypothesis
0,0,English -> Mandarin,eng_Latn,cmn_Hans,qwen2.5,"""We now have 4-month-old mice that are non-dia...",他补充道：“我们现在有 4 个月大没有糖尿病的老鼠，但它们曾经得过该病。”,他补充说：“我们现在有四个月大的非糖尿病小鼠，它们之前都是糖尿病小鼠。”
1,1,English -> Mandarin,eng_Latn,cmn_Hans,qwen2.5,"Dr. Ehud Ur, professor of medicine at Dalhousi...",埃胡德·乌尔博士（新斯科舍省哈利法克斯市达尔豪西大学医学教授，加拿大糖尿病协会临床与科学部门...,多伦多大学哈利法克斯分校的医学教授埃胡德·乌（Ehud Ur）博士警告说，这项研究还处于早期...
2,2,English -> Mandarin,eng_Latn,cmn_Hans,qwen2.5,"Like some other experts, he is skeptical about...",和其他一些专家一样，他对糖尿病能否治愈持怀疑态度。他指出，这些发现与已患有 1 型糖尿病的人无关。,像一些其他专家一样，他对糖尿病能否治愈表示怀疑，并指出这些发现与已经患有1型糖尿病的人无关。
3,3,English -> Mandarin,eng_Latn,cmn_Hans,qwen2.5,"On Monday, Sara Danius, permanent secretary of...",周一，瑞典学院诺贝尔文学委员会常务秘书萨拉·丹尼尔斯在瑞典广播电台的一档节目中向公众宣布，委...,周一，瑞典学院文学奖诺贝尔委员会的常任秘书莎拉·丹尼乌斯在瑞典广播公司的一档广播节目中公开宣...
4,4,English -> Mandarin,eng_Latn,cmn_Hans,qwen2.5,"Danius said, ""Right now we are doing nothing. ...",达尼厄斯说道：“目前我们保持按兵不动。我给他关系最好的合作者打过电话并发送了电子邮件，而且收...,丹尼乌斯说：“我们现在什么都没做。我已经给他最亲近的合作者打了电话并发送了邮件，收到了非常友...


In [12]:
# Convert empty strings to NaN consistently for scoring purposes, and reload Qwen the same way
glm_df = pd.read_csv("glm_only_retry_vllm.csv", keep_default_na=False, na_values=[])
qwen_df = pd.read_csv("qwen_only_vllm.csv", keep_default_na=False, na_values=[])

glm_df["hypothesis"] = glm_df["hypothesis"].replace("", pd.NA)
qwen_df["hypothesis"] = qwen_df["hypothesis"].replace("", pd.NA)

results_df = pd.concat([glm_df, qwen_df], ignore_index=True)
print("Total:", len(results_df))
print(results_df.groupby("model")["hypothesis"].apply(lambda s: s.isna().sum()))

Total: 800
model
glm-4      152
qwen2.5      0
Name: hypothesis, dtype: int64


In [3]:
!pip install -q unbabel-comet sacrebleu

from comet import download_model, load_from_checkpoint
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
INFO:pytorch_lightning.utilities.migration.utils:Lightning au

In [4]:
import gc

def score_comet(df_subset, batch_size=8):
    data = [{"src": r["source"], "mt": r["hypothesis"], "ref": r["reference"]} for _, r in df_subset.iterrows()]
    output = comet_model.predict(data, batch_size=batch_size, gpus=0)  # CPU, to avoid the earlier RAM crash
    gc.collect()
    return output["scores"]

results_df["comet22"] = None
for (direction, model_key), group in results_df.groupby(["direction", "model"]):
    print(f"Scoring {model_key} — {direction}")
    scores = score_comet(group)
    results_df.loc[group.index, "comet22"] = scores

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Scoring glm-4 — Cantonese -> English


Predicting DataLoader 0: 100%|██████████| 13/13 [01:27<00:00,  6.75s/it]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Scoring qwen2.5 — Cantonese -> English


Predicting DataLoader 0: 100%|██████████| 13/13 [01:32<00:00,  7.11s/it]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


...

Predicting DataLoader 0: 100%|██████████| 13/13 [01:24<00:00,  6.48s/it]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Scoring glm-4 — Mandarin -> English


Predicting DataLoader 0: 100%|██████████| 13/13 [01:26<00:00,  6.67s/it]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Scoring qwen2.5 — Mandarin -> English


Predicting DataLoader 0: 100%|██████████| 13/13 [01:29<00:00,  6.91s/it]


In [5]:
import sacrebleu

def chrf_score(hyp, ref):
    if not hyp.strip():
        return 0.0
    return sacrebleu.sentence_chrf(hyp, [ref]).score

results_df["chrf"] = results_df.apply(lambda r: chrf_score(r["hypothesis"], r["reference"]), axis=1)

summary = results_df.groupby(["direction", "model"])[["comet22", "chrf"]].mean().round(2)
summary

comet22   chrf
direction            model                   
Cantonese -> English glm-4    0.697587  37.99
                     qwen2.5  0.848826  54.93
English -> Cantonese glm-4    0.384201   3.32
                     qwen2.5  0.774761  19.39
English -> Mandarin  glm-4    0.441104   9.64
                     qwen2.5  0.877453  36.60
Mandarin -> English  glm-4    0.742323  43.50
                     qwen2.5  0.865288  58.46

In [6]:
cantonese_dir_mask = results_df["tgt_lang"] == "yue_Hant"
cantonese_outputs = results_df[cantonese_dir_mask].copy()

CANTONESE_MARKERS = ["嘅", "咗", "喺", "唔", "佢", "啲", "嘢", "乜"]
def marker_count(text):
    return sum(str(text).count(m) for m in CANTONESE_MARKERS)

cantonese_outputs["output_cantonese_markers"] = cantonese_outputs["hypothesis"].apply(marker_count)
cantonese_outputs["looks_like_cantonese"] = cantonese_outputs["output_cantonese_markers"] > 0
cantonese_outputs["is_empty"] = cantonese_outputs["hypothesis"].str.strip() == ""

print("Among NON-EMPTY English->Cantonese outputs only:")
non_empty = cantonese_outputs[~cantonese_outputs["is_empty"]]
print(non_empty.groupby("model")["looks_like_cantonese"].agg(["mean", "count"]))

Among NON-EMPTY English->Cantonese outputs only:
             mean  count
model                   
glm-4    0.076923     39
qwen2.5  0.160000    100


In [7]:
results_df.to_csv("glm_qwen_translations_scored.csv", index=False)
summary.to_csv("glm_qwen_summary_vllm.csv")
summary

comet22   chrf
direction            model                   
Cantonese -> English glm-4    0.697587  37.99
                     qwen2.5  0.848826  54.93
English -> Cantonese glm-4    0.384201   3.32
                     qwen2.5  0.774761  19.39
English -> Mandarin  glm-4    0.441104   9.64
                     qwen2.5  0.877453  36.60
Mandarin -> English  glm-4    0.742323  43.50
                     qwen2.5  0.865288  58.46